<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/Estadistica/z342_RegLinealNorm_Magicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresión lineal con productos mágicos + normalización

Basado en z403 pero con dos mejoras:

## Imputación de períodos faltantes

La grilla completa es `productos × períodos`. Cuando falta un valor:

- **0** → el producto ya existía (tuvo ventas en algún período anterior) pero no vendió ese mes
- **-1** → el producto todavía no había sido lanzado (sentinel de 'no existe aún')

El -1 le dice al modelo algo cualitativamente distinto al 0: no es ausencia de ventas, es ausencia del producto. El modelo puede aprender que un lag -1 no es informativo.

## Normalización antes de armar los lags

Igual que z341: normalizar cada serie antes de construir los features → el modelo aprende patrones de forma, no de escala.

**Nota**: el -1 (producto no existe) se excluye de la normalización y se mantiene como -1 en los lags.

## Productos mágicos

Igual que z403: se entrena solo con los productos mágicos en `periodo == 201812`, y se predice para todos en `periodo == 201912`.

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3" /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json
mkdir -p /content/buckets/b1/datasets /content/datasets
descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar product_id_apredecir201912.txt

In [ ]:
!pip install uv -q && uv pip install -q kaggle statsmodels

In [ ]:
import os, shutil
import numpy as np
import polars as pl
import polars.selectors as cs
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

PARAM = {
    'experimento':     'RegLinealNormMagicos-01',
    'competencia':     'labo-iii-2026-rosario',
    'semilla':         102191,
    'periodo_train':   201812,   # período donde se toman las observaciones de entrenamiento
    'periodo_predict': 201912,   # período desde donde predecimos t+2
    'lags':            list(range(0, 12)),  # lags 0..11 como features (igual que z403)
    'drive_path':      '/content/buckets/b1/exp/RegLinealNormMagicos',
}

os.makedirs(PARAM['drive_path'], exist_ok=True)

# productos mágicos — igual que z403
PRODUCTOS_MAGICOS = [
    20001, 20002, 20005, 20013, 20033, 20037, 20038, 20043, 20044,
    20045, 20046, 20052, 20055, 20058, 20059, 20069, 20070, 20072, 20073, 20075, 20080,
    20091, 20094, 20099, 20107, 20114, 20120, 20132, 20137, 20139, 20142, 20144, 20146,
    20148, 20151, 20153, 20157, 20158, 20161, 20162, 20166, 20167, 20189, 20198, 20201,
    20202, 20203, 20208, 20226, 20228, 20231, 20233, 20253, 20254, 20256, 20269, 20270,
    20271, 20275, 20276, 20277, 20278, 20288, 20298, 20315, 20317, 20320, 20322, 20335,
    20337, 20338, 20344, 20348, 20350, 20353, 20359, 20385, 20390, 20398, 20402, 20403,
    20406, 20411, 20416, 20417, 20418, 20419, 20421, 20422, 20424, 20428, 20429, 20443,
    20456, 20466, 20469, 20479, 20497, 20500, 20509, 20514, 20517, 20524, 20532, 20549,
    20551, 20560, 20561, 20565, 20568, 20579, 20583, 20585, 20586, 20589, 20599, 20606,
    20614, 20624, 20632, 20642, 20646, 20653, 20655, 20657, 20660, 20661, 20663, 20666,
    20677, 20680, 20684, 20696, 20699, 20713, 20737, 20744, 20745, 20765, 20768, 20773,
    20777, 20786, 20789, 20800, 20807, 20812, 20818, 20830, 20832, 20838, 20847, 20855,
    20863, 20864, 20882, 20883, 20906, 20913, 20914, 20919, 20922, 20925, 20937, 20945,
    20956, 20961, 20965, 20970, 20976, 20986, 20996, 21016, 21038, 21048, 21049, 21077,
    21080, 21088, 21118, 21170, 21200
]

print(f'{len(PRODUCTOS_MAGICOS)} productos mágicos')

# Datos y grilla completa

In [ ]:
dataset      = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator='\t')
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator='\t')

tb_ventas = (
    dataset
    .group_by('product_id', 'periodo')
    .agg(pl.col('tn').sum())
    .join(tb_apredecir, on='product_id', how='inner')
    .sort(['product_id', 'periodo'])
)

productos = tb_apredecir['product_id'].to_list()
periodos  = sorted(tb_ventas['periodo'].unique().to_list())

print(f'{len(productos)} productos, {len(periodos)} períodos')
print(f'Período min: {min(periodos)}, max: {max(periodos)}')

# Imputación — grilla completa con 0 y -1

Construimos la grilla completa `productos × períodos` e imputamos:
- **-1**: períodos anteriores al primer mes con venta del producto (no existía)
- **0**: períodos donde el producto ya existía pero no vendió

In [ ]:
# grilla completa
grilla = pl.DataFrame({
    'product_id': [pid for pid in productos for _ in periodos],
    'periodo':    periodos * len(productos),
})

tb_full = (
    grilla
    .join(tb_ventas.select(['product_id','periodo','tn']),
          on=['product_id','periodo'], how='left')
)

# primer período con venta por producto
primer_periodo = (
    tb_ventas
    .filter(pl.col('tn') > 0)
    .group_by('product_id')
    .agg(pl.col('periodo').min().alias('primer_periodo'))
)

tb_full = tb_full.join(primer_periodo, on='product_id', how='left')

tb_full = tb_full.with_columns(
    pl.when(pl.col('tn').is_not_null())
      .then(pl.col('tn'))                              # valor real
    .when(pl.col('periodo') < pl.col('primer_periodo'))
      .then(pl.lit(-1.0))                              # producto no existía
    .otherwise(pl.lit(0.0))                            # existía pero no vendió
    .alias('tn')
).drop('primer_periodo')

print(f'Grilla: {tb_full.height:,} filas')
print(f'  valores reales: {(tb_full["tn"] > 0).sum():,}')
print(f'  ceros (existía, sin venta): {(tb_full["tn"] == 0).sum():,}')
print(f'  -1 (no existía): {(tb_full["tn"] == -1).sum():,}')
tb_full.head()

# Normalizaciones

In [ ]:
def norm_max(serie: np.ndarray):
    """Divide por el máximo de los valores reales (>0). Escala [0,1].
    Los -1 se mantienen como -1 (sentinel de no-existencia)."""
    reales = serie[serie > 0]
    m = reales.max() if len(reales) > 0 else 1.0
    norm = serie.copy()
    norm[serie >= 0] = serie[serie >= 0] / m   # normaliza 0s y valores reales
    return norm, float(m)                       # -1 queda intacto


def norm_l2(serie: np.ndarray):
    """Norma L2 sobre valores reales (>0). Los -1 se mantienen."""
    reales = serie[serie > 0]
    norma  = float(np.sqrt((reales ** 2).sum())) if len(reales) > 0 else 1.0
    norm   = serie.copy()
    norm[serie >= 0] = serie[serie >= 0] / norma
    return norm, norma


def norm_index(serie: np.ndarray, n_base: int = 3):
    """Divide por la media de los primeros n_base valores positivos. Los -1 se mantienen."""
    positivos = serie[serie > 0]
    base = float(positivos[:n_base].mean()) if len(positivos) > 0 else 1.0
    norm = serie.copy()
    norm[serie >= 0] = serie[serie >= 0] / base
    return norm, base


NORMALIZACIONES = {'sin_norm': None, 'max': norm_max, 'l2': norm_l2, 'index': norm_index}

print('OK')

# Construir tabla de lags normalizada

In [ ]:
def build_lags_norm(tb_full, productos, norm_fn, lags, horizonte=2):
    """
    Para cada producto:
      1. Normaliza la serie (ignorando -1 en el cálculo de la escala)
      2. Construye los lags sobre la serie normalizada
      3. Agrega columna 'clase' = valor normalizado en t+horizonte
      4. Guarda la escala para desnormalizar
    """
    rows      = []
    escalas   = {}

    for pid in productos:
        df     = tb_full.filter(pl.col('product_id') == pid).sort('periodo')
        serie  = df['tn'].to_numpy().astype(float)
        pds    = df['periodo'].to_list()

        if norm_fn is not None:
            serie_norm, escala = norm_fn(serie)
        else:
            serie_norm, escala = serie.copy(), 1.0

        escalas[pid] = escala

        max_lag = max(lags)
        for i, periodo in enumerate(pds):
            # necesitamos max_lag períodos antes y horizonte períodos después
            if i < max_lag or i + horizonte >= len(serie_norm):
                continue
            row = {'product_id': pid, 'periodo': periodo}
            for lag in lags:
                row[f'tn_{lag}'] = float(serie_norm[i - lag])
            row['clase'] = float(serie_norm[i + horizonte])
            rows.append(row)

    return pl.DataFrame(rows), escalas


print('OK')

# Entrenar + predecir + submit para cada normalización

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')


resultados = {}

for nombre_norm, fn in NORMALIZACIONES.items():
    print(f'\n══ norm={nombre_norm} ══')

    # construir lags normalizados
    tb_lags, escalas = build_lags_norm(
        tb_full, productos, fn,
        lags=PARAM['lags'], horizonte=2
    )

    feat_cols = [f'tn_{l}' for l in PARAM['lags']]

    # ── entrenamiento: productos mágicos en periodo_train ──
    dtrain = tb_lags.filter(
        (pl.col('periodo') == PARAM['periodo_train']) &
        (pl.col('product_id').is_in(PRODUCTOS_MAGICOS))
    )
    print(f'  filas de entrenamiento: {dtrain.height}')

    X_train = sm.add_constant(dtrain.select(feat_cols).to_pandas())
    y_train = dtrain['clase'].to_pandas()
    modelo  = sm.OLS(y_train, X_train).fit()
    print(f'  R² entrenamiento: {modelo.rsquared:.4f}')

    # ── predicción: todos los productos en periodo_predict ──
    # solo los que tienen todos los lags completos (sin nulls)
    dfuture = tb_lags.filter(
        pl.col('periodo') == PARAM['periodo_predict']
    ).drop_nulls(subset=feat_cols)
    print(f'  productos con lags completos: {dfuture.height}')

    X_future   = sm.add_constant(dfuture.select(feat_cols).to_pandas())
    pred_norm  = modelo.predict(X_future)

    tb_pred = dfuture.select(['product_id']).with_columns(
        pl.Series('tn_pred_norm', pred_norm.values)
    )

    # desnormalizar
    tb_pred = tb_pred.with_columns(
        pl.struct(['product_id', 'tn_pred_norm'])
        .map_elements(
            lambda r: max(r['tn_pred_norm'] * escalas.get(r['product_id'], 1.0), 0.0),
            return_dtype=pl.Float64
        ).alias('tn')
    ).select(['product_id', 'tn'])

    # fallback: promedio 12 meses para los que no tienen lags completos
    pids_con_pred = set(tb_pred['product_id'].to_list())
    pids_sin_pred = [p for p in productos if p not in pids_con_pred]
    print(f'  fallback (promedio 12m): {len(pids_sin_pred)} productos')

    fallback_rows = []
    for pid in pids_sin_pred:
        serie = tb_full.filter(
            (pl.col('product_id') == pid) &
            (pl.col('tn') >= 0)   # excluir -1
        ).sort('periodo').tail(12)['tn'].to_numpy().astype(float)
        fallback_rows.append({'product_id': pid, 'tn': max(float(serie.mean()), 0.0)})

    tb_final = pl.concat([tb_pred, pl.DataFrame(fallback_rows)]).sort('product_id')

    resultados[nombre_norm] = tb_final

    # guardar y submitear
    archivo = f'reglineal_norm_{nombre_norm}.csv'
    mensaje = f'RegLineal magicos norm={nombre_norm}'
    tb_final.write_csv(archivo)
    shutil.copy(archivo, f"{PARAM['drive_path']}/{archivo}")
    kaggle_submit(PARAM['competencia'], archivo, mensaje)
    print(f'  submitted + guardado: {archivo}')

# Diagnóstico — distribución de predicciones por normalización

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(NORMALIZACIONES), figsize=(14, 4))
colores   = ['gray', 'tomato', 'green', 'purple']

for i, (nombre, tb) in enumerate(resultados.items()):
    vals = tb['tn'].to_numpy()
    cap  = np.percentile(vals, 95)
    axes[i].hist(vals[vals <= cap], bins=40, color=colores[i], edgecolor='white', alpha=0.8)
    axes[i].set_title(f'norm={nombre}\nmedia={vals.mean():.2f}  mediana={np.median(vals):.2f}', fontsize=8)
    axes[i].set_xlabel('tn predicho')

fig.suptitle('Distribución de predicciones 202002 por normalización (hasta p95)', fontsize=10)
plt.tight_layout()
plt.show()

# tabla resumen
print('\nResumen predicciones 202002:')
print(f'{"norm":12s}  {"media":>8}  {"mediana":>8}  {"max":>10}  {"zeros":>6}')
print('-' * 55)
for nombre, tb in resultados.items():
    vals = tb['tn'].to_numpy()
    print(f'{nombre:12s}  {vals.mean():8.3f}  {np.median(vals):8.3f}  {vals.max():10.3f}  {(vals==0).sum():6d}')